# 01 — Data Quality and Cleaning
This notebook documents and executes the production cleaning module. All transformations live in `src/clean.py` so notebook and pipeline results cannot drift.

In [1]:
from pathlib import Path
import sys, json, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
raw = pd.read_csv(ROOT / 'data/raw/jobs_nyc_postings_raw.csv', dtype=str)
raw.shape, raw.columns.tolist()

((2760, 30),
 ['job_id',
  'agency',
  'posting_type',
  'number_of_positions',
  'business_title',
  'civil_service_title',
  'title_classification',
  'title_code_no',
  'level',
  'job_category',
  'full_time_part_time_indicator',
  'career_level',
  'salary_range_from',
  'salary_range_to',
  'salary_frequency',
  'work_location',
  'division_work_unit',
  'job_description',
  'minimum_qual_requirements',
  'preferred_skills',
  'additional_information',
  'to_apply',
  'hours_shift',
  'work_location_1',
  'recruitment_contact',
  'residency_requirement',
  'posting_date',
  'post_until',
  'posting_updated',
  'process_date'])

In [2]:
missing = raw.isna().sum().sort_values(ascending=False).to_frame('missing')
missing.assign(pct=lambda x: (100*x['missing']/len(raw)).round(1)).head(15)

,missing,pct
recruitment_contact,2760,100.0
hours_shift,2437,88.3
additional_information,1927,69.8
to_apply,1639,59.4
preferred_skills,1227,44.5
post_until,39,1.4
minimum_qual_requirements,28,1.0
job_id,0,0.0
agency,0,0.0
posting_updated,0,0.0


In [3]:
from src.clean import clean_dataframe
clean, quality = clean_dataframe(raw)
quality

/Users/balaji305/Documents/Projects/NYC Job Data Analysis/src/clean.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


{'source_rows': 2760,
 'cleaned_rows': 2751,
 'exact_duplicates_removed': 9,
 'duplicate_posting_keys': 12,
 'missing_job_titles': 0,
 'missing_agencies': 0,
 'missing_salary_components': 0,
 'invalid_salary_ranges': 0,
 'salary_midpoint_over_300k': 0,
 'invalid_dates': {'posting_date': 0,
  'post_until': 0,
  'posting_updated': 0,
  'process_date': 0},
 'row_reconciliation_passed': True,
 'fields_cleaned_or_standardized': 20}

In [4]:
clean[['salary_range_from','salary_range_to','salary_frequency','annual_salary_min','annual_salary_max','salary_midpoint','salary_band']].head()

,salary_range_from,salary_range_to,salary_frequency,annual_salary_min,annual_salary_max,salary_midpoint,salary_band
0,100000.00,113500.00,Annual,100000.0,113500.0,106750.0,"$100K-$149,999"
1,46441.00,53407.00,Annual,46441.0,53407.0,49924.0,Under $50K
2,70233.00,80768.00,Annual,70233.0,80768.0,75500.5,"$75K-$99,999"
3,24.88,28.61,Hourly,51750.4,59508.8,55629.6,"$50K-$74,999"
4,62244.00,71581.00,Annual,62244.0,71581.0,66912.5,"$50K-$74,999"


## Decisions
- Preserve the raw snapshot unchanged.
- Remove exact duplicates only; report repeated candidate keys for review.
- Annualize hourly × 2,080 and daily × 260, while retaining source salary fields.
- Coerce malformed dates/numerics to null and count them in QA.
- Retain high salaries as review flags rather than assuming they are errors.